In [21]:
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize, to_hex
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap


# Numeric (Gradient) Style

In [ ]:

def stylize_numeric_ranges(df, columns=None, value_range=(-1, 1), cmap='RdYlGn'):
    """
    Stylizes numeric columns using a color gradient.
    """
    min_val, max_val = value_range
    norm = Normalize(vmin=min_val, vmax=max_val)
    cmap = cm.get_cmap(cmap) if isinstance(cmap, str) else cmap

    def colorize_numeric(val):
        # if pd.isna(val): return ''
        # try:
        rgba = cmap(norm(float(val)))
        return f'background-color: {to_hex(rgba)}'
        # except (ValueError, TypeError):
        #     return ''

    target_cols = columns if columns is not None else df.select_dtypes(include=np.number).columns
    styled = df.style.map(colorize_numeric, subset=list(target_cols))
    return styled


# --- Create Sample Data ---
df = pd.DataFrame({
    "Value": np.linspace(-10, 10, 11),
    "Ratio": np.linspace(-1, 1, 11),
    "isValid": [True, False, True, True, False, True, False, False, True, True, False],
    "Category": ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'A', 'B', 'C', 'A']
})
df['Category'] = df['Category'].astype('category')

custom_rwg = LinearSegmentedColormap.from_list(
    name='RedWhiteGreen',
    colors=['salmon', 'white', 'limegreen']
)
styled = stylize_numeric_ranges(df, columns=['Value'], value_range=(-10, 10), cmap='RdYlGn')
styled


C:\Users\douglas.sgrott_indic\AppData\Local\Temp\ipykernel_25452\2900645331.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(cmap) if isinstance(cmap, str) else cmap


,Value,Ratio,isValid,Category
0,-10.000000,-1.000000,True,A
1,-8.000000,-0.800000,False,B
2,-6.000000,-0.600000,True,A
3,-4.000000,-0.400000,True,C
4,-2.000000,-0.200000,False,B
5,0.000000,0.000000,True,A
6,2.000000,0.200000,False,C
7,4.000000,0.400000,False,A
8,6.000000,0.600000,True,B
9,8.000000,0.800000,True,C


# Boolean Style

In [ ]:
def stylize_boolean(df: pd.DataFrame):
    """
    Applies conditional styling to a DataFrame:
    - True boolean values (Python 'bool' or NumPy 'np.bool_') are colored green.
    - False boolean values (Python 'bool' or NumPy 'np.bool_') are colored red.
    - NaN (None) and empty string '' values are not styled.
    - All other data types are also not styled.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pandas.io.formats.style.Styler: A Styler object with the applied formatting.
    """

    def _apply_boolean_color(val):
        """
        Helper function to determine the background color for a cell based on its value.
        Explicitly checks for Python's 'bool' and NumPy's 'np.bool_'.
        """
        # Handle NaN (None) and empty strings first - no styling
        # if pd.isna(val) or (isinstance(val, str) and val.strip() == ''):
        #     return ''

        # Directly check if the value is an actual boolean type (Python bool or numpy.bool_)
        if isinstance(val, (bool, np.bool_)):
            if val == True:
                return 'background-color: #A3D9B0'  # Light green for True
            elif val == False:
                return 'background-color: #F0A3A3'  # Light red for False
        
        # For any other data type (numbers, other strings, etc.), return no style
        return ''

    # Apply the styling function to each cell in the DataFrame
    return df.style.applymap(_apply_boolean_color)



# Using Custom Pandas Accessor

In [ ]:
# Import pandas and your corrected extension module
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import styler_extensions # This registers the 'style_ext' DataFrame accessor

# --- Create Sample Data ---
df = pd.DataFrame({
    "Value": np.linspace(-10, 10, 11),
    "Ratio": np.linspace(-1, 1, 11),
    "isValid": [True, False, True, True, False, True, False, False, True, True, False],
    "Category": ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'A', 'B', 'C', 'A']
})
df['Category'] = df['Category'].astype('category')
custom_rwg = LinearSegmentedColormap.from_list('RedWhiteGreen', ['salmon', 'white', 'limegreen'])

# --- Use the Fluent, Chainable Interface ---
styled_df = (df.style_ext
    .numeric_ranges(columns=['Value'], value_range=(-10, 10), cmap=custom_rwg)
    .boolean_highlights(columns=['isValid'])
    .categorical_map(columns=['Category'], cmap='Pastel1')
)
# Simply having the object as the last line in a notebook cell will display it correctly
styled_df